In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict

In [ ]:
project_root = Path.cwd()
if not (project_root / "runs").exists():
    project_root = project_root.parent

experiments = [
    "experiment_all_20260528_114458",
    "experiment_all_20260528_114510",
    "experiment_all_20260528_114532",
]

EXPERIMENT_DIRS = [project_root / "runs" / exp for exp in experiments]

# list of all subdirectories, instead of the logs
experiments = []
for exp_dir in EXPERIMENT_DIRS:
    for subdir in exp_dir.iterdir():
        if subdir.is_dir() and not subdir.name.startswith("logs"):
            experiments.append(str(subdir.relative_to(project_root)))

print(len(experiments))
experiments

In [ ]:
import matplotlib

FIGURES_DIR = project_root / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

matplotlib.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times"],
})

In [ ]:
results = [str((project_root / "results" / Path(e).name).relative_to(project_root)) + ".json" for e in experiments]
results

In [ ]:
from tbparse import SummaryReader
from typing import List

raw_dfs: List[pd.DataFrame] = [
    SummaryReader(exp_dir, extra_columns={"wall_time", "dir_name"}).scalars for exp_dir in EXPERIMENT_DIRS
]
df_all = pd.concat(raw_dfs, ignore_index=True)

df_all = df_all.assign(
    wall_clock=pd.to_datetime(df_all["wall_time"], unit="s", utc=True),
    architecture=df_all["dir_name"].str.extract(r"_(cnn|mlp)_")[0],
    train_methods=df_all["dir_name"].str.extract(r"(dd|bp_autodiff)_")[0],
    dataset=df_all["dir_name"].str.extract(r"_(mnist|fashionmnist|cifar\d+)_")[0],
    seed=df_all["dir_name"].str.extract(r"_seed(\d+)")[0].astype(int),
).drop(columns=["wall_time", "dir_name"])

df_all

,step,tag,value,wall_clock,architecture,train_methods,dataset,seed
0,1,autodiff/layer_acc/layer_0/train,0.012200,2026-05-28 11:15:37.269617319+00:00,cnn,bp_autodiff,cifar100,155
1,2,autodiff/layer_acc/layer_0/train,0.016289,2026-05-28 11:15:43.609385014+00:00,cnn,bp_autodiff,cifar100,155
2,3,autodiff/layer_acc/layer_0/train,0.020400,2026-05-28 11:15:49.945050001+00:00,cnn,bp_autodiff,cifar100,155
3,4,autodiff/layer_acc/layer_0/train,0.025133,2026-05-28 11:15:56.272116661+00:00,cnn,bp_autodiff,cifar100,155
4,5,autodiff/layer_acc/layer_0/train,0.030511,2026-05-28 11:16:02.617421389+00:00,cnn,bp_autodiff,cifar100,155
...,...,...,...,...,...,...,...,...
196795,196,dd/mono_ff/loss/val,0.314442,2026-05-29 10:52:13.850655556+00:00,mlp,dd,mnist,1231123
196796,197,dd/mono_ff/loss/val,0.313399,2026-05-29 10:52:16.323612690+00:00,mlp,dd,mnist,1231123
196797,198,dd/mono_ff/loss/val,0.312709,2026-05-29 10:52:18.727905035+00:00,mlp,dd,mnist,1231123
196798,199,dd/mono_ff/loss/val,0.311394,2026-05-29 10:52:21.179484606+00:00,mlp,dd,mnist,1231123


In [5]:
arch_dfs: Dict[str, pd.DataFrame] = {
    arch: sub_df.drop(columns=["architecture"]).reset_index(drop=True)
    for arch, sub_df in df_all.groupby("architecture")
}

df_mlp, df_cnn = arch_dfs["mlp"], arch_dfs["cnn"]
df_mlp["seed"]

0             155
1             155
2             155
3             155
4             155
           ...   
110395    1231123
110396    1231123
110397    1231123
110398    1231123
110399    1231123
Name: seed, Length: 110400, dtype: int64

In [ ]:
mlp_intermediate = (
    df_mlp.groupby(["tag", "train_methods", "dataset", "step"])["value"].agg(mean="mean", std="std").reset_index()
)
mlp_intermediate["tag"].unique()

<ArrowStringArray>
[ 'autodiff/layer_acc/layer_0/train',    'autodiff/layer_acc/layer_0/val',
  'autodiff/layer_acc/layer_1/train',    'autodiff/layer_acc/layer_1/val',
  'autodiff/layer_acc/layer_2/train',    'autodiff/layer_acc/layer_2/val',
  'autodiff/layer_acc/layer_3/train',    'autodiff/layer_acc/layer_3/val',
 'autodiff/layer_loss/layer_0/train',   'autodiff/layer_loss/layer_0/val',
 'autodiff/layer_loss/layer_1/train',   'autodiff/layer_loss/layer_1/val',
 'autodiff/layer_loss/layer_2/train',   'autodiff/layer_loss/layer_2/val',
 'autodiff/layer_loss/layer_3/train',   'autodiff/layer_loss/layer_3/val',
        'autodiff/mono_bp/acc/train',          'autodiff/mono_bp/acc/val',
       'autodiff/mono_bp/loss/train',         'autodiff/mono_bp/loss/val',
        'autodiff/mono_ff/acc/train',          'autodiff/mono_ff/acc/val',
       'autodiff/mono_ff/loss/train',         'autodiff/mono_ff/loss/val',
             'backprop/bp/acc/train',               'backprop/bp/acc/val',
      

In [ ]:
def plot_layers(agg: pd.DataFrame, tags: list, title: str, dataset: str, save_name: str = None):
    fig, ax = plt.subplots(figsize=(8, 5))

    for tag in tags:
        sub = agg[(agg["tag"] == tag) & (agg["dataset"] == dataset)].sort_values("step")
        label = tag.split("/")[-2] if "/" in tag else tag
        (line,) = ax.plot(sub["step"], sub["mean"], label=label)
        ax.fill_between(
            sub["step"],
            sub["mean"] - sub["std"],
            sub["mean"] + sub["std"],
            alpha=0.2,
            color=line.get_color(),
        )

    ax.set_title(f"{title} — {dataset}", fontsize=12)
    ax.set_xlabel("Step")
    ax.set_ylabel("Accuracy (mean ± std)")
    ax.legend(fontsize=9)

    plt.tight_layout()
    if save_name:
        fig.savefig(FIGURES_DIR / f"{save_name}.pdf", bbox_inches="tight")
    plt.show()

In [8]:
# plot_layers(
#     mlp_intermediate,
#     [
#         "autodiff/layer_acc/layer_0/val",
#         "autodiff/layer_acc/layer_1/val",
#         "autodiff/layer_acc/layer_2/val",
#     ],
#     title="MLP autodiff — layer-wise val accuracy",
#     dataset="fashionmnist",
# )

In [9]:
# plot_layers(
#     mlp_intermediate,
#     [
#         "dd/layer_acc/layer_0/val",
#         "dd/layer_acc/layer_1/val",
#         "dd/layer_acc/layer_2/val",
#     ],
#     title="MLP DD — layer-wise val accuracy",
#     dataset="fashionmnist",
# )

In [ ]:
import json
import re


def parse_result(res_path: str):
    stem = Path(res_path).stem
    method = "bp_autodiff" if stem.startswith("bp_autodiff") else "dd"
    arch = re.search(r"_(cnn|mlp)_", stem).group(1)
    dataset = re.search(r"_(mnist|fashionmnist|cifar10|cifar100)_", stem).group(1)
    seed = int(re.search(r"_seed(\d+)$", stem).group(1))
    return method, arch, dataset, seed


rows = []
for res in results:
    method, arch, dataset, seed = parse_result(res)
    data = json.loads((project_root / res).read_text())

    if method == "bp_autodiff":
        rows.append(
            dict(
                method="bp",
                arch=arch,
                dataset=dataset,
                seed=seed,
                sub="ff",
                test_acc=data["bp"]["bp"]["test_acc"],
            )
        )
        rows.append(
            dict(
                method="mf_ad",
                arch=arch,
                dataset=dataset,
                seed=seed,
                sub="ff",
                test_acc=data["autodiff"]["mono_ff"]["test_acc"],
            )
        )
        rows.append(
            dict(
                method="mf_ad",
                arch=arch,
                dataset=dataset,
                seed=seed,
                sub="bp",
                test_acc=data["autodiff"]["mono_bp"]["test_acc"],
            )
        )
    else:
        rows.append(
            dict(
                method="mf_dd",
                arch=arch,
                dataset=dataset,
                seed=seed,
                sub="ff",
                test_acc=data["mono_ff"]["test_acc"],
            )
        )
        rows.append(
            dict(
                method="mf_dd",
                arch=arch,
                dataset=dataset,
                seed=seed,
                sub="bp",
                test_acc=data["mono_bp"]["test_acc"],
            )
        )

raw = pd.DataFrame(rows)

raw.pivot_table(index=["arch", "method", "sub", "dataset"], columns="seed", values="test_acc").sort_index()

seed                          155      200      1231123
arch method sub dataset                                
cnn  bp     ff  cifar10        0.6143   0.6543   0.6406
                cifar100       0.3500   0.3377   0.3458
                fashionmnist   0.8717   0.8669   0.8420
                mnist          0.9669   0.9527   0.9576
     mf_ad  bp  cifar10        0.5359   0.5466   0.5375
                cifar100       0.2678   0.2700   0.2658
                fashionmnist   0.8897   0.8905   0.8839
                mnist          0.9725   0.9710   0.9702
            ff  cifar10        0.5463   0.5590   0.5498
                cifar100       0.2735   0.2772   0.2718
                fashionmnist   0.8860   0.8834   0.8787
                mnist          0.9663   0.9652   0.9648
     mf_dd  bp  cifar10        0.3855   0.3957   0.3951
                cifar100       0.1463   0.1442   0.1460
                fashionmnist   0.8270   0.8347   0.8266
                mnist          0.9575   0.9576   0.9522
            ff  cifar10        0.4260   0.4180   0.4286
                cifar100       0.1730   0.1705   0.1732
                fashionmnist   0.8405   0.8488   0.8417
                mnist          0.9521   0.9525   0.9454
mlp  bp     ff  cifar10        0.4910   0.4855   0.4830
                cifar100       0.2065   0.2115   0.2046
                fashionmnist   0.8801   0.8745   0.8820
                mnist          0.9731   0.9716   0.9731
     mf_ad  bp  cifar10        0.5202   0.5219   0.5256
                cifar100       0.1481   0.1475   0.1518
                fashionmnist   0.8606   0.8536   0.8592
                mnist          0.9467   0.9474   0.9490
            ff  cifar10        0.5212   0.5271   0.5289
                cifar100       0.1887   0.1864   0.1919
                fashionmnist   0.8623   0.8567   0.8610
                mnist          0.9488   0.9514   0.9500
     mf_dd  bp  cifar10        0.3035   0.3024   0.2813
                cifar100       0.0677   0.0710   0.0684
                fashionmnist   0.8257   0.8185   0.8207
                mnist          0.9327   0.9361   0.9311
            ff  cifar10        0.3781   0.3871   0.3715
                cifar100       0.1108   0.1102   0.1139
                fashionmnist   0.8394   0.8370   0.8394
                mnist          0.9339   0.9380   0.9333

In [ ]:
agg = raw.groupby(["arch", "method", "sub", "dataset"])["test_acc"].agg(mean="mean", std="std").reset_index()

DATASETS = ["mnist", "fashionmnist", "cifar10", "cifar100"]


def fmt(mean, std):
    return f"{mean:.2f}+-{std:.2f}"


def make_table(arch: str) -> pd.DataFrame:
    sub_agg = agg[agg["arch"] == arch]

    def get(method, sub, ds):
        s = sub_agg[(sub_agg["method"] == method) & (sub_agg["sub"] == sub) & (sub_agg["dataset"] == ds)]
        return s["mean"].item(), s["std"].item()

    records = []
    for ds in DATASETS:
        bp_mean, bp_std = get("bp", "ff", ds)
        ad_ff_mean, ad_ff_std = get("mf_ad", "ff", ds)
        ad_bp_mean, ad_bp_std = get("mf_ad", "bp", ds)
        dd_ff_mean, dd_ff_std = get("mf_dd", "ff", ds)
        dd_bp_mean, dd_bp_std = get("mf_dd", "bp", ds)
        records.append(
            {
                "dataset": ds,
                "BP": fmt(bp_mean, bp_std),
                "MF+AD (ff)": fmt(ad_ff_mean, ad_ff_std),
                "MF+AD (bp)": fmt(ad_bp_mean, ad_bp_std),
                "MF+DD (ff)": fmt(dd_ff_mean, dd_ff_std),
                "MF+DD (bp)": fmt(dd_bp_mean, dd_bp_std),
                "delta (ff)": f"{dd_ff_mean - ad_ff_mean:+.2f}",
                "delta (bp)": f"{dd_bp_mean - ad_bp_mean:+.2f}",
            }
        )
    return pd.DataFrame(records).set_index("dataset")


table_mlp = make_table("mlp")
table_cnn = make_table("cnn")

print("MLP")
display(table_mlp)
print("\nCNN")
display(table_cnn)

MLP


,BP,MF+AD (ff),MF+AD (bp),MF+DD (ff),MF+DD (bp),delta (ff),delta (bp)
dataset,,,,,,,
mnist,0.97+-0.00,0.95+-0.00,0.95+-0.00,0.94+-0.00,0.93+-0.00,-0.02,-0.01
fashionmnist,0.88+-0.00,0.86+-0.00,0.86+-0.00,0.84+-0.00,0.82+-0.00,-0.02,-0.04
cifar10,0.49+-0.00,0.53+-0.00,0.52+-0.00,0.38+-0.01,0.30+-0.01,-0.15,-0.23
cifar100,0.21+-0.00,0.19+-0.00,0.15+-0.00,0.11+-0.00,0.07+-0.00,-0.08,-0.08



CNN


,BP,MF+AD (ff),MF+AD (bp),MF+DD (ff),MF+DD (bp),delta (ff),delta (bp)
dataset,,,,,,,
mnist,0.96+-0.01,0.97+-0.00,0.97+-0.00,0.95+-0.00,0.96+-0.00,-0.02,-0.02
fashionmnist,0.86+-0.02,0.88+-0.00,0.89+-0.00,0.84+-0.00,0.83+-0.00,-0.04,-0.06
cifar10,0.64+-0.02,0.55+-0.01,0.54+-0.01,0.42+-0.01,0.39+-0.01,-0.13,-0.15
cifar100,0.34+-0.01,0.27+-0.00,0.27+-0.00,0.17+-0.00,0.15+-0.00,-0.10,-0.12
